# Phase 05 — Evaluation & Model Comparison

**Owner:** Nguyễn Duy Khang
Đánh giá và so sánh toàn bộ 5 biến thể mô hình (Random Forest + SMOTENC/ADASYN, XGBoost + SMOTENC/ADASYN, Autoencoder) trên cùng tập test (`X_test`/`y_test`, không SMOTE). Notebook này KHÔNG huấn luyện lại mô hình — chỉ đọc các artifact dự đoán đã lưu ở `reports/*_predictions.pkl` (do Sơn/Cẩm bàn giao) để mọi số liệu đều truy được về một nguồn duy nhất, khớp với `reports/ch5_metrics_recomputed.csv` của Duy.


## 0. Imports & Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # notebooks/ -> project root
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import numpy as np

from src.utils import DATA_PROCESSED_DIR, load_pickle_compat
from src.evaluation import (
    compute_metrics, print_metrics,
    compare_models, save_comparison,
    plot_roc_curves, plot_pr_curves,
    plot_confusion_matrix,
)

REPORTS_DIR = os.path.join("..", "reports")
PROC_DIR = os.path.join("..", DATA_PROCESSED_DIR)

# Nhãn hiển thị ngắn dùng cho tiêu đề/figures — khớp quy ước đã dùng ở
# run_report_metrics.py và plot_confusion_components.py
DISPLAY_LABELS = {
    "Random Forest + SMOTENC": "RF-SMOTENC",
    "Random Forest + ADASYN": "RF-ADASYN",
    "XGBoost + SMOTENC": "XGB-SMOTENC",
    "XGBoost + ADASYN": "XGB-ADASYN",
    "Autoencoder": "Autoencoder",
}


## 1. Load Predictions & Test Labels

Tải nhãn thật `y_test` và dự đoán (`y_pred`, `y_prob`) của cả 5 biến thể từ `reports/`. Các file này do Sơn (RF) và Cẩm (XGBoost, Autoencoder) bàn giao — không tính toán lại ở đây.

In [2]:
def load_processed(name: str):
    return load_pickle_compat(os.path.join(PROC_DIR, f"{name}.pkl"))

def load_report(name: str):
    return load_pickle_compat(os.path.join(REPORTS_DIR, f"{name}.pkl"))

y_test = np.asarray(load_processed("y_test")).astype(int)

rf = load_report("rf_predictions")
xgb = load_report("xgb_predictions")
autoencoder = load_report("autoencoder_predictions")

predictions = {
    "Random Forest + SMOTENC": rf["rf_smote"],
    "Random Forest + ADASYN": rf["rf_adasyn"],
    "XGBoost + SMOTENC": xgb["xgb_smote"],
    "XGBoost + ADASYN": xgb["xgb_adasyn"],
    "Autoencoder": autoencoder,
}

print(f"y_test: {y_test.shape[0]:,} giao dich | fraud = {int(y_test.sum()):,}")
for name, pred in predictions.items():
    print(f"  {name:<26} y_pred={pred['y_pred'].shape}  y_prob={pred['y_prob'].shape}")


y_test: 40,000 giao dich | fraud = 1,643
  Random Forest + SMOTENC    y_pred=(40000,)  y_prob=(40000,)
  Random Forest + ADASYN     y_pred=(40000,)  y_prob=(40000,)
  XGBoost + SMOTENC          y_pred=(40000,)  y_prob=(40000,)
  XGBoost + ADASYN           y_pred=(40000,)  y_prob=(40000,)
  Autoencoder                y_pred=(40000,)  y_prob=(40000,)


## 2. Metrics per Model

In [3]:
metrics_rows = []
for name, pred in predictions.items():
    m = compute_metrics(y_test, pred["y_pred"], pred["y_prob"], model_name=name)
    print_metrics(m)
    metrics_rows.append(m)



  Random Forest + SMOTENC
  Precision (Fraud): 0.9994
  Recall    (Fraud): 0.9951
  F1-Score  (Fraud): 0.9973
  ROC-AUC          : 0.9994
  PR-AUC           : 0.9983


  Random Forest + ADASYN
  Precision (Fraud): 0.9982
  Recall    (Fraud): 0.9951
  F1-Score  (Fraud): 0.9966
  ROC-AUC          : 0.9992
  PR-AUC           : 0.9978


  XGBoost + SMOTENC
  Precision (Fraud): 0.9976
  Recall    (Fraud): 0.9951
  F1-Score  (Fraud): 0.9963
  ROC-AUC          : 0.9993
  PR-AUC           : 0.9969




  XGBoost + ADASYN
  Precision (Fraud): 0.9957
  Recall    (Fraud): 0.9951
  F1-Score  (Fraud): 0.9954
  ROC-AUC          : 0.9994
  PR-AUC           : 0.9976


  Autoencoder
  Precision (Fraud): 0.3822
  Recall    (Fraud): 0.7523
  F1-Score  (Fraud): 0.5069
  ROC-AUC          : 0.9318
  PR-AUC           : 0.5973



## 3. Confusion Matrices (1 per model)

Lưu heatmap riêng cho từng model vào `reports/figures/confusion_matrix_<model>.png`.

In [4]:
for name, pred in predictions.items():
    plot_confusion_matrix(
        y_test, pred["y_pred"],
        model_name=DISPLAY_LABELS[name],
        show=False,
    )
print("Da luu 5 confusion matrix vao reports/figures/")


Da luu 5 confusion matrix vao reports/figures/


## 4. ROC Curves (tất cả model trên 1 plot)

In [5]:
roc_scores = {DISPLAY_LABELS[name]: pred["y_prob"] for name, pred in predictions.items()}
_ = plot_roc_curves(roc_scores, y_test, show=False)


## 5. Precision-Recall Curves (tất cả model trên 1 plot)

In [6]:
_ = plot_pr_curves(roc_scores, y_test, show=False)


## 6. Model Comparison Table

Bảng tổng hợp, sắp xếp theo F1 (fraud) giảm dần, export ra `reports/model_comparison.csv` — bảng chính thức của Phase 05.

In [7]:
comparison_df = compare_models(metrics_rows)
save_comparison(comparison_df, path=os.path.join(REPORTS_DIR, "model_comparison.csv"))
comparison_df


,Model,Precision Fraud,Recall Fraud,F1 Fraud,Roc Auc,Pr Auc
0,Random Forest + SMOTENC,0.9994,0.9951,0.9973,0.9994,0.9983
1,Random Forest + ADASYN,0.9982,0.9951,0.9966,0.9992,0.9978
2,XGBoost + SMOTENC,0.9976,0.9951,0.9963,0.9993,0.9969
3,XGBoost + ADASYN,0.9957,0.9951,0.9954,0.9994,0.9976
4,Autoencoder,0.3822,0.7523,0.5069,0.9318,0.5973


## 7. Kết luận

- **Best model: Random Forest + SMOTENC** — F1(fraud) = 0.9973, ROC-AUC = 0.9994, Precision = 0.9994, Recall = 0.9951. Đây là mức hiệu năng tốt nhất trong 5 biến thể, và cũng nhất quán với số liệu Chương 5 (Duy) tính từ cùng nguồn artifact.
- **4 biến thể có giám sát (RF/XGBoost × SMOTENC/ADASYN)** đều đạt F1 > 0.995 và ROC-AUC > 0.999 — khác biệt giữa chúng rất nhỏ (xem thêm phân tích khoảng tin cậy ở `reports/ch6_prevalence_projection.csv` của Duy, cho thấy khác biệt này chưa đủ ý nghĩa thống kê để xếp hạng chắc chắn).
- **Autoencoder** (không giám sát, train chỉ trên giao dịch bình thường) có Recall = 0.7523 và ROC-AUC = 0.9318 nhưng Precision chỉ 0.3822 — đánh đổi hợp lý cho một mô hình không cần nhãn fraud khi huấn luyện, phù hợp để phát hiện các kiểu gian lận mới chưa từng thấy.
- **Trade-off tổng thể:** nếu ưu tiên độ chính xác cao nhất trên dữ liệu đã biết → chọn RF-SMOTENC; nếu cần khả năng phát hiện anomaly tổng quát không phụ thuộc nhãn → Autoencoder là lựa chọn bổ trợ.

**Output đã tạo:**
- `reports/model_comparison.csv`
- `reports/figures/roc_curves_all.png`
- `reports/figures/pr_curves_all.png`
- `reports/figures/confusion_matrix_rf-smotenc.png`, `confusion_matrix_rf-adasyn.png`, `confusion_matrix_xgb-smotenc.png`, `confusion_matrix_xgb-adasyn.png`, `confusion_matrix_autoencoder.png`
